In [ ]:
INDEX = 8  # Select which questions in the benchmark to test (0-based)
DATA_SOURCE = "biomedical"

# Setup

In [ ]:
import sys
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv("../../../")

sys.path.append("../../../src")

from processor.model.interface.impl.gpt import GPT
from processor.model.llm_message import LLMMessage, Role
from processor.model.option import LLMOption
import bm25s
import json
import Stemmer

In [ ]:
stemmer = Stemmer.Stemmer("english")
gpt = GPT("gpt-4o")

In [ ]:
def write_jsonl(filepath, data, append=False):
    """
    Write a list of JSON-serializable objects to a JSONL file.

    Args:
        filepath (str): Path to the output file.
        data (list): List of Python dictionaries or objects to write.
        append (bool): If True, append to existing file. Otherwise, overwrite.
    """
    mode = 'a' if append else 'w'
    with open(filepath, mode, encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [ ]:
def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl(f"../../../benchmark/benchmark_{DATA_SOURCE}.jsonl")
INITIAL_PROMPT = benchmark[INDEX]["interactive_initial_prompt"]
benchmark[INDEX]

In [ ]:
def get_format_to_gpt(results):
    format_str = "SYSTEM OUTPUT:"
    for result in results[0]:
        format_str += f"\n{result}"
    return format_str

def get_initial_prompt_to_chatgpt(domain: str, question: str):
    domain_expert_desc = f"a {domain} domain expert"
    if domain == "archeology":
        domain_expert_desc = "a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration"
    return f"""You are simulating {domain_expert_desc}, who is interacting with a basic table discovery system to explore insights from an enterprise dataset.

This system supports static table lookup: it returns the contents (rows or description) of one or more tables based on your description. However:
- The system does not infer your deeper intent.
- The system does not combine, transform, or analyze data for you.
- The system does not explain its reasoning—it simply returns table contents for you to explore.

Your task is to gradually explore and refine your question about some aspect of the data. You do not begin with a precise question — your curiosity evolves step-by-step based on the tables you receive. You will gradually refine your question by examining the contents of the tables returned.

In this scenario:
- The system already has access to an internal dataset.
- You are familiar with the domain and have seen similar datasets before.
- You are not uploading new datasets or asking if data exists — you assume it does.

Here is a possible eventual goal (you do not know this at the start, and you may or may not reach it):

{question}

Your behavior should reflect:
- You are familiar with the domain but must infer relevant relationships from static tables.
- You refine your question step-by-step depending on what the returned tables show.
- You may explore tangents or ask for different table contents in later turns.
- You will only reach the specific question above if you deduce it from the table contents, which may take multiple turns.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {INITIAL_PROMPT}"""

In [ ]:
retriever = bm25s.BM25.load(f"indices/keyword-index-{DATA_SOURCE}", load_corpus=True)

# Evaluation

In [ ]:
ITERATION_LIMIT = 15

gpt_init_prompt = get_initial_prompt_to_chatgpt(
    DATA_SOURCE, benchmark[INDEX]["original_direct_question"]
)
gpt_messages = [LLMMessage(role=Role.SYSTEM.value, content=gpt_init_prompt)]
curr_user_prompt = INITIAL_PROMPT
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")
for iteration in tqdm(range(ITERATION_LIMIT)):
    query_tokens = bm25s.tokenize(
        curr_user_prompt, stemmer=stemmer, show_progress=False
    )
    # Prevent out-of-memory errors for biomedical dataset
    results, _ = retriever.retrieve(query_tokens, k=10 if DATA_SOURCE != "biomedical" else 5, show_progress=False)
    format_to_gpt = get_format_to_gpt(results)
    if iteration == 0:
        gpt_messages[0]['content'] += f"\n{format_to_gpt}"
    else:
        gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
    updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
    gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
    if updated_user_prompt.startswith("YOU:"):
        updated_user_prompt = updated_user_prompt[4:]
        updated_user_prompt = updated_user_prompt.strip()
    curr_user_prompt = updated_user_prompt
    print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

In [ ]:
write_jsonl(f"benchmark_data/{DATA_SOURCE}_{INDEX+1}.jsonl", gpt_messages, True)

## Convergence Checking

In [ ]:
benchmark_data = read_jsonl(f"benchmark_data/{DATA_SOURCE}_{INDEX+1}.jsonl")
filtered_benchmark_data = [i for i in benchmark_data if i['role'] == 'assistant']
filtered_benchmark_data[0]
actual_benchmark = read_jsonl(f"../../../benchmark/benchmark_{DATA_SOURCE}.jsonl")[INDEX]
actual_benchmark["original_direct_question"]
def get_eval_prompt(hidden, bench_data):
    return (f"""You are evaluating whether a simulated domain expert has successfully converged on a target information need.

Below is the target (hidden) goal question:
---
{hidden}
---

Below is the most recent query (or set of queries) made by the simulated expert:
---
{bench_data}
---

Does the user's most recent query express the same information need as the hidden goal, either literally or in semantically equivalent terms?

Respond with only one of the following labels:
- CONVERGED (if CONVERGED, tell me where is the point it converges)
- NOT CONVERGED (explain a little why)""")

In [ ]:
eval_message = [
    LLMMessage(
        role=Role.SYSTEM.value,
        content=get_eval_prompt(
            actual_benchmark["original_direct_question"],
            filtered_benchmark_data
        )
    )
]
gpt.chat(eval_message)